# Policy Gradient: CartPole

REINFORCE (Williams, 1992) on a pure Idris CartPole environment. The agent
learns a policy that maps observations (cart position, velocity, pole angle,
angular velocity) to actions (push left/right).

This is the library's only reinforcement learning example, demonstrating that
idris-ml handles non-standard training loops beyond supervised learning.

**CLI equivalent:** `make example-reinforce` (2000 epochs, converges to 200.0 return)

## Architecture

A simple 2-layer MLP:
```
Linear(4 -> 128) -> Tanh -> Linear(128 -> 2)
```

Output: 2 logits (push left / push right). Action sampled via
`categoricalSample` from log-softmax probabilities.

In [ ]:
:t categoricalSample

## CartPole Environment

The CartPole physics are implemented in pure Idris (no gym/gymnasium dependency).
Gymnasium-compatible constants: gravity=9.8, pole length=0.5, force=10.0,
dt=0.02. Episode terminates when:
- Cart position |x| > 2.4
- Pole angle |theta| > 12 degrees
- 200 steps reached (maximum return)

The environment is deterministic given the same random seed.

## REINFORCE Algorithm

1. Collect batch of episodes using current policy
2. For each step: log_prob * (G_t - baseline)
   - `G_t` = discounted return from step t
   - baseline = mean return across batch (variance reduction)
3. Sum losses, backpropagate, update policy

The key tensor operation: `prim__select` picks the log-probability of the
chosen action, maintaining the autograd graph for backpropagation.

## Model Construction

We can build the policy network interactively. The CartPole environment
and REINFORCE training loop are in the compiled example
(`src/Example/Reinforce.idr`).

In [ ]:
:exec do { srand 42;
  ll1 <- linearLayer {i=4, o=64};
  ll2 <- linearLayer {i=64, o=2};
  model <- pure (autoName (ll1 ~> tanhLayer ~> OutputLayer ll2));
  putStrLn ("Model: " ++ show model);
  putStrLn ("Param count: " ++ show (networkParamCount model)) }

Training requires the CartPole environment and custom `epochRL` function
(defined in `src/Example/Reinforce.idr`). The training loop:

```idris
opt <- pure (nativeAdamGlobalClip 0.001 0.9 0.999 1.0e-8 1.0)
(trained, epochs, loss) <- runTraining
  (\m, d => epochRL opt 0.99 m d)
  (genBatch 10) (simpleConfig 2000) model
```

The loss value is the negative mean episode return. As training progresses
it approaches -200.0 (perfect balance for all 200 timesteps).

Run via CLI: `make example-reinforce --epochs 2000`

## Scaling Up

For full convergence (200.0 greedy return on all backends):
```bash
make example-reinforce --epochs 2000 --lr 0.001 --batch 10
```

The loss reported during training is the negative mean return.
As training progresses, it should approach -200.0 (perfect balance
for all 200 timesteps).

## PyTorch Comparison

```python
# Standard REINFORCE
probs = F.softmax(model(obs), dim=-1)
action = torch.multinomial(probs, 1)
log_prob = torch.log(probs[action])

# After episode:
loss = -sum(log_prob * (G_t - baseline))
loss.backward()
optimizer.step()
```

In idris-ml, the rollout loop and loss computation are in pure Idris,
but the tensor operations (`prim__logSoftmax`, `prim__select`) go through
the C backend and participate in autograd.

See `pytorch/torch_ref/scripts/reinforce.py` for the full reference.

Next: [SeqClassify](seq_classify.ipynb) — 1D convolutions for waveform classification.